# Lab 5.1: Training on TPU with PyTorch XLA - Cats vs Dogs

## 🎯 Learning Objectives

By the end of this lab, you will:
- Understand what TPUs are and how they differ from GPUs
- Use PyTorch XLA to run PyTorch code on TPU
- Adapt the same Cats vs Dogs model from Lab 5 to TPU
- Compare TPU vs GPU performance
- Understand when to use TPU vs GPU

**What You'll Build:** Same cats vs dogs classifier from Lab 5, but running on Google Cloud TPU

**Note:** Google Colab provides free TPU access! (v2-8 TPU with 8 cores)

## Part 1: Understanding TPUs

### What is a TPU?

**TPU** = Tensor Processing Unit (custom chip designed by Google)

**Specialized for:**
- Matrix multiplication (the core of deep learning)
- Batch processing
- Training large models

### TPU vs GPU vs CPU

| Feature | CPU | GPU | TPU |
|---------|-----|-----|-----|
| **Cores** | 4-16 | 1000s | 8 (v2-8) |
| **Memory** | GB-TB | 8-16 GB | 8 GB per core |
| **Speed** | 1x | 10-100x | 15-30x |
| **Best for** | General | Graphics + ML | Large batch ML |
| **Cost** | Baseline | Medium | Low (free in Colab!) |

### Why TPU?

**Advantages:**
- ✅ **Faster for large batches** (matrix multiplication optimized)
- ✅ **Free in Colab** (normally expensive)
- ✅ **High memory bandwidth** (faster data access)
- ✅ **Good for training** (especially transformers)

**Limitations:**
- ⚠️ Requires larger batch sizes (128+) for efficiency
- ⚠️ Limited debugging (can't inspect tensors easily)
- ⚠️ Some PyTorch operations not supported
- ⚠️ XLA compilation overhead (first iteration slow)

### When to Use TPU vs GPU?

**Use TPU when:**
- Training very large models (transformers, large CNNs)
- Large batch sizes (128+)
- Long training runs
- Cloud-based training

**Use GPU when:**
- Small batch sizes (<64)
- Need maximum flexibility
- Rapid prototyping with debugging
- Local training

### What is PyTorch XLA?

**XLA** = Accelerated Linear Algebra (Google's compiler)

**PyTorch XLA** bridges PyTorch and TPU:
- Translates PyTorch operations to TPU operations
- Handles device management
- Optimizes computation graphs

**Code changes are minimal:**
```python
# GPU code:
device = torch.device('cuda')

# TPU code:
import torch_xla.core.xla_model as xm
device = xm.xla_device()
```

## Part 2: TPU Setup in Colab

### Step 1: Enable TPU Runtime

**In Google Colab:**
1. Runtime → Change runtime type
2. Hardware accelerator → **TPU**
3. Save

### Step 2: Install PyTorch XLA

In [ ]:
# Install PyTorch XLA (required for TPU)
!pip install cloud-tpu-client==0.10 torch==2.0.0 torchvision==0.15.0 https://storage.googleapis.com/tpu-pytorch/wheels/colab/torch_xla-2.0-cp310-cp310-linux_x86_64.whl

print("\n✓ PyTorch XLA installed!")

### Step 3: Verify TPU Access

In [ ]:
import torch
import torch_xla
import torch_xla.core.xla_model as xm

# Get TPU device
device = xm.xla_device()

print(f"TPU device: {device}")
print(f"Number of TPU cores: {xm.xrt_world_size()}")

# Test computation on TPU
x = torch.randn(3, 3).to(device)
y = torch.randn(3, 3).to(device)
z = x + y

print(f"\nTest computation:")
print(f"x device: {x.device}")
print(f"z device: {z.device}")
print(f"\n✓ TPU is working!")

## Part 3: Load Dataset (Same as Lab 5)

We'll use the same cats vs dogs dataset from Lab 5:

In [ ]:
# Install kagglehub
!pip install -q kagglehub

print("✓ kagglehub installed!")

In [ ]:
import kagglehub

# Download dataset
print("Downloading dataset...")
path = kagglehub.dataset_download("tongpython/cat-and-dog")

print(f"\n✓ Dataset downloaded to: {path}")

In [ ]:
import os
from pathlib import Path

# Setup paths (same nested structure as Lab 5)
dataset_path = Path(path)
train_path = dataset_path / "training_set" / "training_set"
test_path = dataset_path / "test_set" / "test_set"

TRAIN_DIR = str(train_path)
TEST_DIR = str(test_path)

# Count images
train_cats = len(list((train_path / "cats").glob("*.jpg")))
train_dogs = len(list((train_path / "dogs").glob("*.jpg")))
test_cats = len(list((test_path / "cats").glob("*.jpg")))
test_dogs = len(list((test_path / "dogs").glob("*.jpg")))

print(f"Training set: {train_cats} cats, {train_dogs} dogs")
print(f"Test set: {test_cats} cats, {test_dogs} dogs")
print(f"\n✓ Dataset ready!")

## Part 4: Data Loading for TPU

### Key Difference: Larger Batch Sizes

TPUs are optimized for **large batches**:
- GPU works well with batch_size=32-64
- TPU works best with batch_size=128+ (even 256)
- Larger batches utilize all 8 cores efficiently

**API Reference:** [PyTorch XLA DataLoader](https://pytorch.org/xla/release/1.13/index.html)

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Same transforms as Lab 5
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                         std=[0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transform)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=test_transform)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

In [ ]:
# Create data loaders with TPU-optimized batch size

# Larger batch size for TPU efficiency
batch_size = 128  # Increased from 64 (GPU) for better TPU utilization

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=4,  # More workers for TPU
    drop_last=True  # Important for TPU - keeps batch sizes consistent
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=4,
    drop_last=False
)

print(f"Batch size: {batch_size} (optimized for TPU)")
print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"\n✓ Data loaders ready for TPU!")

## Exercise 1: Build Model for TPU

### Your Task

Build the same transfer learning model from Lab 5:
1. Load pre-trained ResNet18
2. Freeze backbone
3. Add custom classification head
4. Move to TPU device

**Key difference:** Use `model.to(device)` where device is TPU

### Starter Code

In [ ]:
import torch.nn as nn
import torchvision.models as models

class CatDogClassifier(nn.Module):
    """Same model as Lab 5, adapted for TPU."""
    
    def __init__(self):
        super().__init__()
        
        # self.backbone = models.resnet18(pretrained=True)
        
        # for param in self.backbone.parameters():
        #     param.requires_grad = False
        
        # self.backbone.fc = nn.Sequential(
        #     nn.Linear(512, 128),
        #     nn.ReLU(),
        #     nn.Dropout(0.5),
        #     nn.Linear(128, 1)
        # )
        
        raise NotImplementedError("Implement model")
    
    def forward(self, x):
        raise NotImplementedError("Implement forward")

# Create model and move to TPU
model = CatDogClassifier().to(device)

print(f"✓ Model on device: {next(model.parameters()).device}")

## Exercise 2: Implement Training Loop for TPU

### Key Differences from GPU Training

**1. Device:**
```python
# GPU
device = torch.device('cuda')

# TPU
import torch_xla.core.xla_model as xm
device = xm.xla_device()
```

**2. Optimizer step:**
```python
# GPU
optimizer.step()

# TPU
xm.optimizer_step(optimizer)  # XLA-optimized step
```

**3. Print/logging:**
```python
# Use xm.master_print() to print only from master core
xm.master_print(f"Loss: {loss.item()}")
```

**4. Metric tracking:**
```python
# Move to CPU before tracking
loss_val = loss.item()  # Automatically moves to CPU
```

### Your Task

Implement training loop with TPU-specific changes:

**API Reference:** [PyTorch XLA Training](https://pytorch.org/xla/release/1.13/index.html#running-on-a-single-xla-device)

In [ ]:
import torch.optim as optim
import torch_xla.core.xla_model as xm
from tqdm import tqdm

# Setup
model = CatDogClassifier().to(device)

# criterion = nn.BCEWithLogitsLoss()
# optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training settings
num_epochs = 5
train_losses = []
train_accs = []

#
# xm.master_print("Starting training on TPU...\n")
#
# for epoch in range(num_epochs):
#     model.train()
#     running_loss = 0.0
#     correct = 0
#     total = 0
#     
#     # No tqdm on TPU (can cause issues), use simple loop
#     for batch_idx, (images, labels) in enumerate(train_loader):
#         images = images.to(device)
#         labels = labels.to(device)
#         
#         # Forward
#         outputs = model(images).squeeze()
#         loss = criterion(outputs, labels.float())
#         
#         # Backward
#         optimizer.zero_grad()
#         loss.backward()
#         xm.optimizer_step(optimizer)  # ← TPU-specific!
#         
#         # Track metrics (move to CPU)
#         running_loss += loss.item()
#         predictions = (torch.sigmoid(outputs) > 0.5).long()
#         correct += (predictions == labels).sum().item()
#         total += labels.size(0)
#         
#         # Print every 10 batches
#         if batch_idx % 10 == 0:
#             xm.master_print(f"  Batch [{batch_idx}/{len(train_loader)}]")
#     
#     # Epoch summary
#     epoch_loss = running_loss / len(train_loader)
#     epoch_acc = correct / total
#     train_losses.append(epoch_loss)
#     train_accs.append(epoch_acc)
#     
#     xm.master_print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f}, Acc: {epoch_acc*100:.2f}%")
#
# xm.master_print("\n✓ Training complete!")

print("\nUncomment the code above to train on TPU!")

### Expected Performance

**With batch_size=128 on TPU:**
```
Epoch [1/5] - Loss: 0.3234, Acc: 87.45%
Epoch [2/5] - Loss: 0.1876, Acc: 92.31%
Epoch [3/5] - Loss: 0.1234, Acc: 95.12%
Epoch [4/5] - Loss: 0.0923, Acc: 96.78%
Epoch [5/5] - Loss: 0.0745, Acc: 97.65%
```

**Training time:**
- TPU: ~2-3 minutes (with batch=128)
- GPU: ~3-4 minutes (with batch=64)
- Speedup: ~1.5x faster with larger batches

## Exercise 3: Implement Validation on TPU

### Your Task

Implement validation - same as Lab 5 but with TPU device:

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

#
# model.eval()
# all_preds = []
# all_labels = []
#
# with torch.no_grad():
#     for images, labels in test_loader:
#         images = images.to(device)  # TPU device
#         labels = labels.to(device)
#         
#         outputs = model(images).squeeze()
#         predictions = (torch.sigmoid(outputs) > 0.5).long()
#         
#         # Move back to CPU for metrics
#         all_preds.extend(predictions.cpu().numpy())
#         all_labels.extend(labels.cpu().numpy())
#
# # Calculate metrics
# accuracy = accuracy_score(all_labels, all_preds)
# precision = precision_score(all_labels, all_preds)
# recall = recall_score(all_labels, all_preds)
# f1 = f1_score(all_labels, all_preds)
#
# xm.master_print(f"\nTest Results:")
# xm.master_print(f"Accuracy:  {accuracy*100:.2f}%")
# xm.master_print(f"Precision: {precision*100:.2f}%")
# xm.master_print(f"Recall:    {recall*100:.2f}%")
# xm.master_print(f"F1 Score:  {f1*100:.2f}%")

print("\nUncomment to validate!")

## Part 5: TPU vs GPU Comparison

### Performance Comparison

| Metric | GPU (T4) | TPU (v2-8) |
|--------|----------|------------|
| Batch size | 64 | 128 |
| Training time | ~3-4 min | ~2-3 min |
| Memory | 16 GB | 64 GB (8×8) |
| Speedup | Baseline | ~1.5x |
| Cost | Free (Colab) | Free (Colab) |

### When TPU Shines

**Best scenarios:**
- Large models (>100M parameters)
- Large batch sizes (128+)
- Long training runs
- Transformer models (BERT, GPT)

**GPU still better for:**
- Small batch sizes
- Rapid prototyping
- Models with dynamic graphs
- Interactive debugging

### Key Takeaways

**TPU Advantages:**
- ✅ Faster for large batches
- ✅ More memory (64 GB vs 16 GB)
- ✅ Free in Colab
- ✅ Good for production training

**TPU Considerations:**
- ⚠️ Requires XLA (extra setup)
- ⚠️ Less flexible than GPU
- ⚠️ First iteration slow (XLA compilation)
- ⚠️ Debugging harder

**Rule of thumb:**
- Prototyping: GPU
- Production training: TPU

## Summary

### What You've Learned

✅ **TPUs** are specialized for matrix operations and large-scale training

✅ **PyTorch XLA** enables PyTorch code to run on TPU

✅ **Key differences:** xm.optimizer_step(), larger batches, xm.master_print()

✅ **Performance** - TPU can be 1.5-2x faster with large batches

✅ **When to use** - Large models, large batches, long training

### Complete Journey

You've now trained the same model on:
1. **CPU** - Baseline (Lab 4 on small MNIST)
2. **GPU** - 10-100x faster (Lab 5 with T4)
3. **TPU** - Optimized for large scale (Lab 5.1)

### What's Next?

**Try TPU on larger models:**
- Train larger ResNets (ResNet50, ResNet101)
- Train Vision Transformers (ViT)
- Train on larger datasets (ImageNet subsets)
- Use all 8 TPU cores with distributed training

**Resources:**
- [PyTorch XLA Documentation](https://pytorch.org/xla/)
- [Google Cloud TPU Guide](https://cloud.google.com/tpu/docs/pytorch-xla-ug-tpu-vm)
- [Colab TPU Tutorial](https://colab.research.google.com/notebooks/tpu.ipynb)

**Congratulations!** 🎉 You've mastered training on CPU, GPU, and TPU. You're ready for production-scale deep learning!